# Умный помощник по научным статьям (PaperAssistant)

## 📝 Описание проекта

Многоагентный помощник по научным статьям на базе фреймворка HelloAgents: поиск литературы, резюме статей, генерация цитирования, редактирование текста и создание плана.

### Информация об авторе
- Имя: chengH425
- GitHub: @chengH425
- Дата: 2026-07-20

---
## Часть 1: Настройка окружения

In [ ]:
# Установка зависимостей (можно пропустить, если уже установлены)
# !pip install -q hello-agents python-dotenv

In [ ]:
# Импорт основных библиотек
import os
import sys
import json
from datetime import datetime
from typing import Dict, Any, List

# Совместимость консоли Windows с кодировкой UTF-8
sys.stdout.reconfigure(encoding='utf-8')

from dotenv import load_dotenv
from hello_agents import (
    HelloAgentsLLM, SimpleAgent, ReflectionAgent,
    PlanSolveAgent, ToolRegistry, Config
)
from hello_agents.tools import Tool, ToolParameter, ToolResponse, ToolStatus

# Загрузка переменных окружения
load_dotenv()

print("Настройка окружения завершена!")
print(f"   LLM Model: {os.getenv('LLM_MODEL_ID', 'Qwen/Qwen2.5-72B-Instruct')}")

---
## Часть 2: Определение инструментов

In [ ]:
# Импорт пользовательских инструментов из модуля src/ (9 инструментов)
from src.citation_tool import CitationTool
from src.literature_tool import LiteratureSearchTool
from src.aminer_tool import AminerSearchTool
from src.openalex_tool import OpenAlexSearchTool
from src.pubmed_tool import PubMedSearchTool
from src.crossref_tool import CrossRefSearchTool
from src.arxiv_tool import ArxivSearchTool
from src.pdf_tool import PDFExtractTool

# Инструмент статистического анализа текста (лёгкий, определяется напрямую)
class TextAnalysisTool(Tool):
    """Инструмент статистического анализа текста"""
    def __init__(self):
        super().__init__(
            name="text_analysis",
            description="Анализ статистики текста: количество символов, абзацев, предложений и т.д."
        )
    def run(self, parameters: Dict[str, Any]) -> ToolResponse:
        text = parameters.get("text", "")
        if not text:
            return ToolResponse.error(code="INVALID_PARAM", message="Текст не может быть пустым")
        chinese_chars = sum(1 for c in text if '\u4e00' <= c <= '\u9fff')
        english_words = len([w for w in text.split() if any(c.isalpha() for c in w)])
        sentences_cn = len([s for s in text.replace('!', '。').replace('?', '。').split('。') if s.strip()])
        sentences_en = len([s for s in text.replace('!', '.').replace('?', '.').split('.') if s.strip()])
        paragraphs = len([p for p in text.split('\n') if p.strip()])
        result = {
            "Всего символов": len(text), "Китайских символов": chinese_chars,
            "Английских слов": english_words, "Предложений (кит.)": sentences_cn,
            "Предложений (англ.)": sentences_en, "Абзацев": paragraphs,
            "Оценка времени чтения (мин)": round((chinese_chars / 400 + english_words / 200), 1)
        }
        return ToolResponse.success(
            text=json.dumps(result, ensure_ascii=False, indent=2), data=result)
    def get_parameters(self) -> List[ToolParameter]:
        return [ToolParameter(name="text", type="string",
                             description="Текст для анализа", required=True)]

print("Определение инструментов завершено! Всего 9 инструментов:")
print("   Поиск (6): Semantic Scholar / AMiner / OpenAlex / PubMed / CrossRef / arXiv")
print("   Обработка (3): CitationTool / PDFExtractTool / TextAnalysisTool")

---
## Часть 3: Создание агентов

В системе **4 парадигмы агентов** работают совместно:

| Агент | Парадигма | Задачи |
|--------|------|------|
| SearchAgent | SimpleAgent | Поиск литературы и систематизация информации |
| SummaryAgent | SimpleAgent | Резюме содержания статей |
| PolishAgent | ReflectionAgent | Редактирование статей (итеративная оптимизация через саморефлексию) |
| OutlineAgent | PlanSolveAgent | Структурированное создание плана статьи |

In [ ]:
# Создание экземпляра LLM (отключение trace-логов для совместимости с кодировкой Windows)
llm = HelloAgentsLLM()
config = Config(trace_enabled=False)

# Создание реестра инструментов, регистрация всех 9 инструментов
tool_registry = ToolRegistry()
tool_registry.register_tool(LiteratureSearchTool())   # Semantic Scholar
tool_registry.register_tool(AminerSearchTool())       # AMiner (китайские статьи)
tool_registry.register_tool(OpenAlexSearchTool())     # OpenAlex
tool_registry.register_tool(PubMedSearchTool())       # PubMed
tool_registry.register_tool(CrossRefSearchTool())     # CrossRef
tool_registry.register_tool(ArxivSearchTool())        # arXiv
tool_registry.register_tool(CitationTool())
tool_registry.register_tool(PDFExtractTool())
tool_registry.register_tool(TextAnalysisTool())

# ========================================
# Агент 1: Помощник по поиску литературы (SimpleAgent + 6 инструментов поиска)
# ========================================
search_system_prompt = """Вы — эксперт по поиску академической литературы. У вас есть 6 инструментов поиска:

- literature_search: Semantic Scholar, все дисциплины (рекомендуется в первую очередь)
- aminer_search: AMiner, китайские академические статьи (предпочтительно для китайской литературы)
- openalex_search: OpenAlex, открытый доступ к статьям
- pubmed_search: PubMed, биомедицинская область
- crossref_search: CrossRef, метаданные журнальных статей
- arxiv_search: arXiv, препринты по CS/математике/физике

Правила:
1. Обязательно используйте указанный пользователем инструмент поиска для получения реальных данных
2. При ошибке вызова инструмента сообщайте об ошибке напрямую, не выдумывайте статьи
3. Анализируйте и рекомендуйте на основе реальных результатов"""

search_agent = SimpleAgent(
    name="Помощник по поиску литературы", llm=llm,
    system_prompt=search_system_prompt, config=config
)
for name in ["literature_search", "aminer_search", "openalex_search",
             "pubmed_search", "crossref_search", "arxiv_search"]:
    search_agent.add_tool(tool_registry.get_tool(name))

# ========================================
# Агент 2: Помощник по резюме статей (SimpleAgent)
# ========================================
summary_system_prompt = """Вы — эксперт по рецензированию академических статей, умеете быстро извлекать ключевую информацию.

Для заданного содержания статьи составьте структурированный отчёт-резюме:

## Информация о статье
- Название, авторы, год публикации, журнал/конференция

## Исследовательская проблема
- Какую ключевую проблему решает эта статья?

## Методы и новизна
- Какие методы/модели/алгоритмы использованы?
- В чём ключевая новизна по сравнению с существующими работами?

## Эксперименты и результаты
- На каких наборах данных проводились эксперименты?
- Основные результаты и метрики производительности

## Вклад и ограничения
- Основной вклад статьи (1–3 пункта)
- Ограничения или нерешённые вопросы

## Идеи и направления развития
- Какие идеи эта статья даёт для вашего направления исследований?
- Какие направления можно исследовать дальше?

Пишите отчёт на русском языке, профессиональные термины оставляйте на английском."""

summary_agent = SimpleAgent(
    name="Помощник по резюме статей",
    llm=llm,
    system_prompt=summary_system_prompt,
    config=config
)

# ========================================
# Агент 3: Помощник по редактированию статей (SimpleAgent, режим многоходового диалога)
# ========================================
# В Web UI создаётся отдельный экземпляр через фабричную функцию create_polish_agent()
# При каждом новом диалоге создаётся новый Agent, history естественно накапливает контекст
# Правила: сохранять смысл, улучшать формулировки, помнить историю правок

# ========================================
# Агент 4: Помощник по созданию плана (SimpleAgent, режим многоходового диалога)
# ========================================
# В Web UI создаётся отдельный экземпляр через фабричную функцию create_outline_agent()
# При каждом новом диалоге создаётся новый Agent, поддерживается постепенная доработка
# Правила: структурированная декомпозиция, поддержка последующих инструкций вроде "детализировать третью главу"

# ========================================
# Агент 5: Помощник по написанию статей (SimpleAgent + 6 инструментов поиска, режим многоходового диалога)
# ========================================
# В Web UI создаётся отдельный экземпляр через фабричную функцию create_paper_writer_agent()
# Регистрируются все 6 инструментов поиска для цитирования реальной литературы
# Правила: писать по главам согласно плану, перед цитированием обязательно искать, запрет на выдуманные источники
# Поддержка последующих инструкций вроде "напиши вторую главу", "добавь больше про transformer"

print("Агенты созданы!")
print("   1. search_agent  - Помощник по поиску литературы (SimpleAgent + 6 инструментов поиска)")
print("   2. summary_agent - Помощник по резюме статей (SimpleAgent)")
print("   3. polish_agent  - Помощник по редактированию (режим диалога, Web UI)")
print("   4. outline_agent - Помощник по созданию плана (режим диалога, Web UI)")
print("   5. writer_agent  - Помощник по написанию статей (режим диалога + инструменты поиска, Web UI)")

---
## Часть 4: Демонстрация функций

### 📚 Демо 1: Поиск литературы

In [ ]:
print("=" * 60)
print("📚 Демо 1: Поиск литературы")
print("=" * 60)

search_query = "Прогресс в применении больших языковых моделей в программной инженерии"
print(f"\n🔍 Тема поиска: {search_query}\n")

search_result = search_agent.run(search_query)
print(search_result)
print("\n" + "=" * 60)

### 📝 Демо 2: Резюме статьи

In [ ]:
print("=" * 60)
print("📝 Демо 2: Резюме статьи")
print("=" * 60)

# Пример аннотации статьи
sample_paper = """
Название статьи: Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks
Авторы: Patrick Lewis, Ethan Perez, Aleksandra Piktus, et al.
Публикация: NeurIPS 2020

Аннотация: Large pre-trained language models have been shown to store factual knowledge 
in their parameters, and achieve state-of-the-art results when fine-tuned on 
downstream NLP tasks. However, their ability to access and precisely manipulate 
knowledge is still limited, leading to factual errors and hallucinations. We 
introduce Retrieval-Augmented Generation (RAG), a general-purpose fine-tuning 
approach that combines pre-trained parametric and non-parametric memory for 
language generation. RAG models retrieve relevant documents from a dense vector 
index and condition the generation on both the input and retrieved documents. 
We evaluate RAG on a diverse set of NLP tasks including open-domain QA, abstractive 
question answering, and fact verification, achieving state-of-the-art results. 
Our analysis shows that RAG generates more specific, diverse, and factual language 
compared to parametric-only models.
"""

print(f"\n📄 Статья для резюме: {sample_paper[:80]}...\n")

summary_result = summary_agent.run(
    f"Составьте структурированное резюме следующего содержания статьи:\n\n{sample_paper}"
)
print(summary_result)
print("\n" + "=" * 60)

### 📎 Демо 3: Генерация цитирования

In [ ]:
print("=" * 60)
print("📎 Демо 3: Генерация цитирования в нескольких форматах")
print("=" * 60)

# Тестовые данные статей
test_papers = [
    {
        "title": "Attention Is All You Need",
        "authors": "Vaswani, A., Shazeer, N., Parmar, N., Uszkoreit, J., Jones, L., Gomez, A. N., Kaiser, L., Polosukhin, I.",
        "journal": "Advances in Neural Information Processing Systems",
        "year": "2017",
        "volume": "30",
        "pages": "5998-6008"
    },
    {
        "title": "BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding",
        "authors": "Devlin, J., Chang, M. W., Lee, K., Toutanova, K.",
        "journal": "Proceedings of the 2019 Conference of the North American Chapter of the ACL",
        "year": "2019",
        "volume": "1",
        "pages": "4171-4186"
    },
    {
        "title": "Chain-of-Thought Prompting Elicits Reasoning in Large Language Models",
        "authors": "Wei, J., Wang, X., Schuurmans, D., Bosma, M., Ichter, B., Xia, F., Chi, E., Le, Q., Zhou, D.",
        "journal": "Advances in Neural Information Processing Systems",
        "year": "2022",
        "volume": "35",
        "pages": "24824-24837"
    }
]

formats = ["gbt7714", "apa", "mla"]
format_names = {"gbt7714": "GB/T 7714 (китайский стандарт)", "apa": "APA 7th", "mla": "MLA 9th"}

for i, paper in enumerate(test_papers, 1):
    print(f"\n--- Статья {i}: {paper['title'][:50]}... ---")
    for fmt in formats:
        paper["format"] = fmt
        response = tool_registry.execute_tool("citation_generator", json.dumps(paper))
        print(f"\n  [{format_names[fmt]}]:")
        print(f"  {response.text}")

print("\n" + "=" * 60)

### ✍️ Демо 4: Редактирование статьи

In [ ]:
print("=" * 60)
print("✍️ Демо 4: Многоходовое редактирование статьи в диалоге")
print("=" * 60)

# Имитация многоходового диалога
round1 = """Отредактируйте следующий академический абзац, чтобы он лучше соответствовал нормам академического письма:

In this paper, we propose a new method to solve the problem. Our method is very 
good and it works better than other methods. We did a lot of experiments."""

print(f"\n📝 【Раунд 1】Пользователь: {round1[:80]}...\n")

# Первый раунд: начальное редактирование
polish_agent1 = SimpleAgent(
    name="Редактирование", llm=llm, config=config,
    system_prompt="Вы — редактор академических текстов. После редактирования объясните внесённые изменения."
)
result1 = polish_agent1.run(f"Пользователь: {round1}\nАссистент: ")
print(result1[:500])
print("\n---\n")

# Второй раунд: дальнейшая оптимизация с учётом контекста
round2 = "Сделайте второе предложение более академичным, используйте более точную лексику"
print(f"📝 【Раунд 2】Пользователь: {round2}\n")
polish_agent2 = SimpleAgent(
    name="Редактирование", llm=llm, config=config,
    system_prompt="Вы — редактор академических текстов. Помните контекст и продолжайте правки на основе уже сделанного."
)
result2 = polish_agent2.run(
    f"Пользователь: {round1}\nАссистент: {result1}\nПользователь: {round2}\nАссистент: "
)
print(result2[:500])
print("\n" + "=" * 60)

### 📊 Демо 5: Создание плана статьи

In [ ]:
print("=" * 60)
print("📊 Демо 5: Создание плана статьи (PlanSolveAgent)")
print("=" * 60)

outline_topic = "Проектирование и реализация многоагентной системы совместной работы на базе больших языковых моделей"
print(f"\n📋 Тема статьи: {outline_topic}\n")
print("🔄 PlanSolveAgent разбивает задачу и формирует план...\n")

outline_result = outline_agent.run(
    f"Составьте подробный структурированный план для следующей темы статьи: {outline_topic}"
)
print(outline_result)
print("\n" + "=" * 60)

### 📝 Демо 6: Написание статьи (по плану + реальная литература)

Демонстрация пошагового написания статьи по плану с вызовом инструментов поиска для цитирования реальных источников.

In [ ]:
print("=" * 60)
print("📝 Демо 6: Написание статьи (по плану + реальная литература)")
print("=" * 60)

# Имитация плана статьи
outline = """План статьи: Обзор анализа медицинских изображений на основе глубокого обучения
Глава 1 Введение
Глава 2 Медицинские изображения и основы глубокого обучения
Глава 3 Методы анализа медицинских изображений на основе CNN
Глава 4 Сравнение экспериментов и оценка производительности
Глава 5 Перспективы и вызовы"""

print(f"\n📋 Заданный план:\n{outline}\n")
print("🔄 Создание агента для написания статей (с возможностью поиска литературы)...\n")

# Создание агента для написания с инструментами поиска
writer_llm = HelloAgentsLLM()
writer_agent = SimpleAgent(
    name="Написание статей", llm=writer_llm, config=config,
    system_prompt="""Вы — эксперт по написанию академических статей. Можете вызывать инструменты поиска литературы для нахождения реальных статей.
При цитировании опирайтесь только на результаты поиска, категорически запрещено выдумывать источники."""
)
# Регистрация инструментов поиска для цитирования реальной литературы
for name in ["literature_search", "aminer_search"]:
    writer_agent.add_tool(tool_registry.get_tool(name))

# Первый раунд: написание введения
result1 = writer_agent.run(
    f"По следующему плану напишите введение первой главы (около 300 слов)."
    f"Если нужно цитировать литературу, используйте инструмент literature_search для поиска реальных статей:\n{outline}"
)
print("--- Глава 1 Введение ---")
print(result1[:600])
print("...\n")

# Второй раунд: раздел о методах, требуется цитирование литературы
result2 = writer_agent.run(
    "Напишите содержание третьей главы, опишите как минимум два распространённых метода анализа "
    "медицинских изображений на основе CNN. Используйте aminer_search или literature_search "
    "для поиска релевантных статей и цитируйте их в тексте."
)
print("--- Глава 3 Методы на основе CNN ---")
print(result2[:600])
print("...\n")

print("=" * 60)

---
## Часть 5: Демонстрация инструмента анализа текста

In [ ]:
print("=" * 60)
print("📈 Дополнительное демо: Статистический анализ текста")
print("=" * 60)

sample_text = """
Большие языковые модели (Large Language Models, LLMs) в последние годы достигли прорывных результатов.
Предобученные языковые модели серии GPT демонстрируют мощные возможности в задачах обработки естественного языка.
Однако при практическом применении LLM по-прежнему сталкиваются с галлюцинациями, недостаточными способностями к рассуждению и другими проблемами.
В этой статье обзорны исследования последних лет по усилению способностей LLM к рассуждению,
с акцентом на принципы и эффективность методов подсказок Chain-of-Thought, Tree-of-Thought и др.
Исследования показывают, что структурированное проектирование путей рассуждения значительно повышает результаты LLM на сложных задачах.
We systematically review recent advances in enhancing LLM reasoning capabilities.
Our analysis covers both prompting-based methods and training-based approaches.
The results demonstrate that structured reasoning significantly improves performance.
"""

response = tool_registry.execute_tool("text_analysis", json.dumps({"text": sample_text}))
print(response.text)
print("\n" + "=" * 60)

---
## Часть 6: Итоги и перспективы

### ✅ Реализованные функции

1. **Поиск литературы** — интеллектуальный помощник на базе SimpleAgent с систематической стратегией поиска
2. **Резюме статей** — структурированное извлечение ключевой информации (проблема, метод, вклад, ограничения)
3. **Генерация цитирования** — пользовательский CitationTool поддерживает форматы GB/T 7714 / APA / MLA
4. **Редактирование статей** — итеративная оптимизация через саморефлексию на базе ReflectionAgent
5. **Создание плана** — структурированный план статьи на базе PlanSolveAgent

### 🔧 Технические особенности

- **Совместная работа агентов разных парадигм**: объединены SimpleAgent, ReflectionAgent и PlanSolveAgent
- **Пользовательская система инструментов**: на базе Tool + ToolParameter реализованы CitationTool и TextAnalysisTool
- **Структурированный вывод**: у каждой функции есть чёткий вывод в формате Markdown

### 🚧 Встреченные трудности

- Задержки LLM API нестабильны, детерминированные вычисления на стороне инструментов частично компенсируют это
- Правила форматов цитирования сложны, пограничные случаи APA/MLA требуют дальнейшей проработки

### 🔮 Направления будущих улучшений

- [ ] Подключение arXiv API для поиска статей в реальном времени
- [ ] Поддержка загрузки и разбора PDF
- [ ] Добавление анализа на плагиат
- [ ] Создание веб-интерфейса на Gradio
- [ ] Внедрение механизма дебатов между агентами для повышения качества рецензирования